## 1. Imports

# Bengaluru House Price Prediction

Predicts house prices in Bengaluru using a linear regression model, trained on the Kaggle *Bengaluru House Data* dataset.

**Pipeline:** data cleaning -> feature engineering -> outlier removal -> one-hot encoding -> model selection (GridSearchCV) -> prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Data

In [ ]:
df = pd.read_csv("data/Bengaluru_House_Data.csv")

In [ ]:
df.head()

In [ ]:
df.shape

## 3. Data Cleaning
Drop unused columns and handle missing values.

In [ ]:
df1=df.drop(['society','area_type','availability','balcony'],axis=1)

In [ ]:
df1.head()

In [ ]:
df1.info()

In [ ]:
df1.isna().sum()

In [ ]:
df1.dropna(inplace=True)
df1.isna().sum()

## 4. Feature Engineering
Parse `size` into a numeric `Bedroom` count and normalize `total_sqft` into a single numeric column.

In [ ]:
df1['size'].unique()

In [ ]:
df1['Bedroom']=df1['size'].apply(lambda x:int(x.split(' ')[0]))

In [ ]:
df1.head()

In [ ]:
df1['Bedroom'].unique()

In [ ]:
df1["total_sqft"].unique()

In [ ]:
def not_a_number(x):
    try:
        float(x)
        return False
    except:
        return True         

In [ ]:
df1[df1["total_sqft"].apply(lambda x:not_a_number(x))]

In [ ]:
def corr_sqft_column(x):
    x = str(x).strip()

    if '-' in x:
        tokens = x.split('-')
        return (float(tokens[0]) + float(tokens[1])) / 2
        
    elif 'Sq. Meter' in x:
        num = float(x.split('S')[0])
        return num * 10.7639
        
    elif 'Sq. Yards' in x:
        num = float(x.split('S')[0])
        return num * 9.0
        
    elif 'Acres' in x:
        num = float(x.split('A')[0])
        return num * 43560.0
        
    elif 'Cents' in x:
        num = float(x.split('C')[0])
        return num * 435.6
        
    elif 'Guntha' in x:
        num = float(x.split('G')[0])
        return num * 1089.0
        
    elif 'Grounds' in x:
        num = float(x.split('G')[0])
        return num * 2400.0

    elif 'Perch' in x:
        num = float(x.split('P')[0].strip())
        return num * 272.25
        
    else:
        return float(x)

In [ ]:
df1['sqft']=df1["total_sqft"].apply(lambda x:corr_sqft_column(x))

## 5. Location Grouping
Bucket rare locations (<10 listings) into an `others` category to reduce dimensionality.

In [ ]:
df1["location"].nunique()

In [ ]:
location_stats=df1.groupby("location")["location"].count()

In [ ]:
location_stats[location_stats<10]

In [ ]:
less_than_10_location=location_stats[location_stats<10]

In [ ]:
len(less_than_10_location)

In [ ]:
df1["location"]=df1["location"].apply(lambda x:'others' if x in less_than_10_location else x)

In [ ]:
df1["location"].nunique()

## 6. Outlier Removal (I): Price per Sqft
Drop implausibly small properties and statistical outliers in price per sqft.

In [ ]:
df1["price per sqft"]=(df1["price"]*100000)/df1["sqft"]

In [ ]:
df1.head()

In [ ]:
df_new = df1[~(df1.sqft / df1.Bedroom < 300)]

m = np.mean(df_new['price per sqft'])
st = np.std(df_new['price per sqft'])
max_limit = m + st
min_limit = m - st

df_new = df_new[(df_new['price per sqft'] >= min_limit) & (df_new['price per sqft'] <= max_limit)].reset_index(drop=True)

print("Global Filtering Shape:", df_new.shape)

In [ ]:
df_new1=df_new.copy()

## 7. Visualization Helper
Scatter plot to compare 2 BHK vs 3 BHK pricing in a given location.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def scatter_plot(location, df):
    # Filters data points matching the target location and specific bedrooms
    bhk2bhk = df[(df.location == location) & (df.Bedroom == 2)]
    bhk3bhk = df[(df.location == location) & (df.Bedroom == 3)]
    
    plt.figure(figsize=(15, 10))
    # Using 'sqft' and 'price' columns from your dataset
    plt.scatter(bhk2bhk.sqft, bhk2bhk.price, color='blue', label='2 BHK', s=50)
    plt.scatter(bhk3bhk.sqft, bhk3bhk.price, marker='+', color='green', label='3 BHK', s=50)
    
    plt.xlabel("Price")
    plt.ylabel("Sqft")
    plt.title(location)
    plt.legend()
    plt.show()

In [ ]:
scatter_plot("Hebbal",df_new)

## 8. Outlier Removal (II): BHK Price Anomalies
Remove listings priced below the average for the prior BHK tier in the same location.

In [ ]:
df_stats = df_new.groupby(["location", "Bedroom"])["price per sqft"].agg(['mean', 'count']).reset_index()
df_stats = df_stats.rename(columns={"mean": "mean_price_per_sqft_of_prev_bhk"})

df_new['prev_bhk_target'] = df_new['Bedroom'] - 1

df_merged = pd.merge(
    df_new, 
    df_stats, 
    how='left', 
    left_on=["location", "prev_bhk_target"], 
    right_on=["location", "Bedroom"]
)


drop_condition = (
    df_merged["price per sqft"] < df_merged["mean_price_per_sqft_of_prev_bhk"]
) & (df_merged["count"] > 5)

df_filtered = df_merged[~drop_condition].copy()

df_filtered = df_filtered.reset_index(drop=True)

In [ ]:
df_filtered

In [ ]:
df_filtered.shape

## 9. Outlier Removal (III): Bathroom Count
Inspect distributions and drop listings with implausible bathroom counts.

In [ ]:
plt.hist(df_filtered["price per sqft"],rwidth=0.8)
plt.xlabel("price per sqft")
plt.ylabel("Count")
plt.title("Distribution")

In [ ]:
df_filtered[df_filtered.bath > 10]

In [ ]:
plt.hist(df_filtered["bath"],rwidth=0.8)
plt.xlabel("bathrrom")
plt.ylabel("Count")
plt.title("Distribution")

In [ ]:
df_final=df_filtered[~(df_filtered.bath > df_filtered.Bedroom_x+2)].copy()

## 10. Finalize Dataset
Drop helper columns and rename to final feature names.

In [ ]:
df_final.drop(columns=["size","total_sqft","price per sqft","prev_bhk_target","Bedroom_y","mean_price_per_sqft_of_prev_bhk","count"],axis=1,inplace=True)

In [ ]:
df_final.rename(columns={"Bedroom_x":"bhk","sqft":"total_sqft"},inplace=True)

In [ ]:
df_final.head()

## 11. One-Hot Encode Location

In [ ]:
dummies=pd.get_dummies(df_final.location)
dummies.head()

In [ ]:
df_final=pd.concat([df_final,dummies.drop(columns=["others"])],axis=1)

In [ ]:
df_final

In [ ]:
df_final.shape

## 12. Train / Test Split

In [ ]:
X=df_final.drop(columns=["price","location"],axis=1)

In [ ]:
y=df_final.price

## 13. Model Selection
Compare Linear Regression, Lasso, and Decision Tree Regressor via `GridSearchCV`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso
# 1. Swap the classifier for the regressor
from sklearn.tree import DecisionTreeRegressor 

model_and_param = {
    'lr_reg': {
        'model': LinearRegression(),
        'param': {
            'fit_intercept': [True, False]
        }
    },
    'lasso': {
        'model': Lasso(),
        'param': {
            'alpha': [1, 2],
            'selection': ['random', 'cyclic']
        }
    },
    'DeTrReg': { # Changed name to reflect Regressor
        'model': DecisionTreeRegressor(),
        'param': {
            # 2. These are now perfectly valid for a Regressor!
            'criterion': ['squared_error', 'friedman_mse'], 
            'splitter': ['best', 'random'] 
        }
    }
}

In [ ]:
scores=[]
for model,mp in model_and_param.items():
    clf=GridSearchCV(mp["model"],mp['param'],cv=5)
    clf.fit(X,y)
    scores.append({"Model":model,"Best_param":clf.best_params_,"Best_Score":clf.best_score_})
result=pd.DataFrame(scores,columns=["Model","Best_param","Best_Score"])
result

## 14. Fit Final Model

In [ ]:
lr_reg=LinearRegression(fit_intercept=True)
lr_reg.fit(X,y)

## 15. Prediction Function

In [ ]:
def predict_price(bath,bhk,total_sqft,location):
    loc_index=X.columns.get_loc(location)
    x=np.zeros(len(X.columns))
    x[0]=bath
    x[1]=bhk
    x[2]=total_sqft
    x[loc_index]=1
    input_df = pd.DataFrame([x], columns=X.columns)
    return lr_reg.predict(input_df)

In [ ]:
predict_price(2,2,1500,"Vijayanagar")

## 16. Save Model
Model is saved to `model/lr_reg_model.pickle`.

In [ ]:
import os
import joblib

os.makedirs("model", exist_ok=True)
joblib.dump(lr_reg, "model/lr_reg_model.pickle")